In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(128),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]),
    'val': transforms.Compose([
        transforms.Resize(128),
        transforms.CenterCrop(128),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
}
data_dir =data_dir = r"C:\Users\Harsh\Downloads\archive (3)"
subdirs = [d.name for d in os.scandir(data_dir) if d.is_dir()]
print("Found folders:", subdirs)

if "train" in subdirs and "val" in subdirs:
    print("")
    image_datasets = {
        x: datasets.ImageFolder(os.path.join(data_dir, x), transform=data_transforms[x])
        for x in ['train', 'val']
    }
    dataloaders = {
        x: DataLoader(image_datasets[x], batch_size=32, shuffle=True)
        for x in ['train', 'val']
    }
else:
    print("⚠️ No train/val folders found — splitting automatically.")
    full_dataset = datasets.ImageFolder(data_dir, transform=data_transforms['train'])

    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

    dataloaders = {
        'train': DataLoader(train_dataset, batch_size=32, shuffle=True),
        'val': DataLoader(val_dataset, batch_size=32, shuffle=False)
    }
model = models.resnet18(pretrained=True)
# Modify final layer for binary classification
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.0003)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
for epoch in range(5):  # start small for testing
    model.train()
    running_loss = 0.0

    for inputs, labels in dataloaders['train']:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(dataloaders['train'])
    print(f"Epoch {epoch+1}, Training Loss: {avg_loss:.4f}")
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in dataloaders['val']:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}, Validation Accuracy: {accuracy:.2f}%\n")

print(correct,total)


Found folders: ['ants', 'bees']
⚠️ No train/val folders found — splitting automatically.


C:\Users\Harsh\anaconda3\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Harsh\anaconda3\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Harsh/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|█████████████████████████████████████████████████████████████████████████████| 44.7M/44.7M [00:05<00:00, 8.26MB/s]


Epoch 1, Training Loss: 0.7697
Epoch 1, Validation Accuracy: 50.00%

Epoch 2, Training Loss: 0.5816
Epoch 2, Validation Accuracy: 77.27%

Epoch 3, Training Loss: 0.5501
Epoch 3, Validation Accuracy: 77.27%

Epoch 4, Training Loss: 0.6039
Epoch 4, Validation Accuracy: 81.82%

Epoch 5, Training Loss: 0.5176
Epoch 5, Validation Accuracy: 86.36%

19 22
